# Taller - BCI Simulado: Señales Mentales Artificiales para Control Visual

Este notebook implementa un flujo básico de BCI simulado:
1. Carga de señal EEG desde CSV (o generación sintética si no existe).
2. Visualización temporal de amplitud vs tiempo.
3. Filtrado pasa banda para Alpha (8-12 Hz) y Beta (12-30 Hz).
4. Cálculo de potencia en banda y definición de una condición de control.
5. Simulación visual interactiva que responde al nivel de activación.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt, welch
from pathlib import Path

plt.style.use('seaborn-v0_8-whitegrid')

: 

## 1) Carga de CSV o generación de señal EEG sintética

Si existe `eeg_data.csv` en esta carpeta, se carga ese archivo.
Si no existe, se genera una señal sintética y se guarda como CSV para reutilizarla.

In [ ]:
# Parametros base de muestreo
fs = 250  # Hz
duration = 20  # segundos
n = fs * duration
t = np.arange(n) / fs

csv_path = Path('eeg_data.csv')

if csv_path.exists():
    df = pd.read_csv(csv_path)
    required_cols = {'time_s', 'eeg_uV'}
    if not required_cols.issubset(df.columns):
        raise ValueError(
            f'El CSV debe contener las columnas {required_cols}, encontradas: {set(df.columns)}'
        )
    t = df['time_s'].to_numpy()
    eeg = df['eeg_uV'].to_numpy()
    fs = int(round(1 / np.mean(np.diff(t))))
    print(f'CSV cargado: {csv_path.resolve()} | Muestras: {len(df)} | fs estimada: {fs} Hz')
else:
    rng = np.random.default_rng(42)

    # Componentes base: alpha y beta + artefacto lento + ruido
    alpha = 25 * np.sin(2 * np.pi * 10 * t)
    beta = 10 * np.sin(2 * np.pi * 20 * t)
    drift = 15 * np.sin(2 * np.pi * 0.5 * t)
    noise = rng.normal(0, 8, size=t.shape)

    # Segmento con mayor beta para simular un estado de atencion
    attention_boost = ((t > 8) & (t < 14)).astype(float)
    beta_dynamic = beta + attention_boost * 15 * np.sin(2 * np.pi * 18 * t)

    eeg = alpha + beta_dynamic + drift + noise

    df = pd.DataFrame({'time_s': t, 'eeg_uV': eeg})
    df.to_csv(csv_path, index=False)
    print(f'Senal sintetica generada y guardada en: {csv_path.resolve()}')

df.head()

## 2) Visualizacion de la señal en el tiempo

In [ ]:
plt.figure(figsize=(14, 4))
plt.plot(t, eeg, color='tab:blue', linewidth=1)
plt.title('EEG (uV) vs Tiempo (s)')
plt.xlabel('Tiempo (s)')
plt.ylabel('Amplitud (uV)')
plt.tight_layout()
plt.show()

## 3) Filtrado pasa banda Alpha (8-12 Hz) y Beta (12-30 Hz)

In [ ]:
def bandpass_filter(signal, fs, low, high, order=4):
    nyq = 0.5 * fs
    low_n = low / nyq
    high_n = high / nyq
    b, a = butter(order, [low_n, high_n], btype='band')
    return filtfilt(b, a, signal)

eeg_alpha = bandpass_filter(eeg, fs, 8, 12)
eeg_beta = bandpass_filter(eeg, fs, 12, 30)

plt.figure(figsize=(14, 5))
plt.plot(t, eeg_alpha, label='Alpha 8-12 Hz', linewidth=1.2)
plt.plot(t, eeg_beta, label='Beta 12-30 Hz', linewidth=1.0, alpha=0.85)
plt.title('Senales filtradas por banda')
plt.xlabel('Tiempo (s)')
plt.ylabel('Amplitud filtrada (uV)')
plt.legend()
plt.tight_layout()
plt.show()

## 4) Potencia en banda y condición artificial de control

Se calcula la potencia por ventanas deslizantes y se define un indicador de atencion.
Condicion de ejemplo: `atencion = potencia_beta > umbral_beta`.

In [ ]:
def band_power_welch(signal_window, fs, fmin, fmax):
    freqs, psd = welch(signal_window, fs=fs, nperseg=min(len(signal_window), fs))
    mask = (freqs >= fmin) & (freqs <= fmax)
    return np.trapezoid(psd[mask], freqs[mask])

win_sec = 1.0
step_sec = 0.2
win = int(win_sec * fs)
step = int(step_sec * fs)

times_w = []
alpha_powers = []
beta_powers = []

for start in range(0, len(eeg) - win + 1, step):
    end = start + win
    chunk = eeg[start:end]

    ap = band_power_welch(chunk, fs, 8, 12)
    bp = band_power_welch(chunk, fs, 12, 30)

    alpha_powers.append(ap)
    beta_powers.append(bp)
    times_w.append(t[start + win // 2])

alpha_powers = np.array(alpha_powers)
beta_powers = np.array(beta_powers)
times_w = np.array(times_w)

beta_threshold = np.percentile(beta_powers, 65)
attention_state = beta_powers > beta_threshold

print(f'Umbral Beta (percentil 65): {beta_threshold:.2f}')
print(f'Porcentaje de activacion: {attention_state.mean() * 100:.1f}%')

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

ax[0].plot(times_w, alpha_powers, label='Potencia Alpha', color='tab:green')
ax[0].plot(times_w, beta_powers, label='Potencia Beta', color='tab:red')
ax[0].axhline(beta_threshold, color='tab:red', linestyle='--', alpha=0.7, label='Umbral Beta')
ax[0].set_ylabel('Potencia')
ax[0].set_title('Dinamica de potencia en bandas')
ax[0].legend()

ax[1].step(times_w, attention_state.astype(int), where='mid', color='black')
ax[1].set_yticks([0, 1])
ax[1].set_yticklabels(['Inactivo', 'Activo'])
ax[1].set_xlabel('Tiempo (s)')
ax[1].set_ylabel('Estado')
ax[1].set_title('Condicion artificial de atencion')

plt.tight_layout()
plt.show()

## 5) Simulacion visual interactiva en tiempo real (matplotlib)

Regla visual:
- Si `atencion = 1`: el circulo cambia a color rojo y se mueve mas rapido a la derecha.
- Si `atencion = 0`: el circulo es azul y avanza lentamente.

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

fig, ax = plt.subplots(figsize=(10, 2.5))
ax.set_xlim(0, 10)
ax.set_ylim(-1, 1)
ax.set_yticks([])
ax.set_xlabel('Espacio visual')
ax.set_title('Control visual por activacion BCI simulada')

point = ax.scatter([0.5], [0], s=300, c='tab:blue')
state_txt = ax.text(0.02, 0.85, '', transform=ax.transAxes, fontsize=12)

x_positions = [0.5]

def update(i):
    active = bool(attention_state[i])
    speed = 0.18 if active else 0.05
    new_x = x_positions[-1] + speed
    if new_x > 9.5:
        new_x = 0.5
    x_positions.append(new_x)

    color = 'tab:red' if active else 'tab:blue'
    point.set_offsets([[new_x, 0]])
    point.set_color(color)

    label = 'ATENCION ALTA' if active else 'ATENCION BAJA'
    state_txt.set_text(f't={times_w[i]:.1f}s | {label}')
    return point, state_txt

anim = FuncAnimation(fig, update, frames=len(times_w), interval=int(step_sec * 1000), blit=False)
plt.close(fig)
HTML(anim.to_jshtml())